# MISO DA-RT Spread Forecasting — LSTM Model
**DTE Energy Trading | Michigan Hub**

Forecasts hourly Day-Ahead minus Real-Time price spreads to generate
virtual bid signals (DEC/INC) for the MISO energy market.

- **Train:** 2023–2024 (in-sample)
- **Test:**  2025 (out-of-sample)
- **Architecture:** 2-layer LSTM, 168-hour lookback (1 week)
- **Target:** DA-RT spread ($/MWh)

## 1. Imports & Configuration

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

# Paths
DB_PATH  = r"C:\文件\MISO_Trading_Analysis\database\miso_trading.db"
OUT_PATH = r"C:\文件\MISO_Trading_Analysis\outputs"

# Model hyperparameters
SEQUENCE_LENGTH = 168   # 1-week lookback
HIDDEN_SIZE     = 128
NUM_LAYERS      = 2
DROPOUT         = 0.2
BATCH_SIZE      = 64
EPOCHS          = 30
LEARNING_RATE   = 0.001

MODEL_PATH   = r"C:\文件\MISO_Trading_Analysis\outputs\lstm_model.pth"
SCALER_X_PATH = r"C:\文件\MISO_Trading_Analysis\outputs\scaler_X.pkl"
SCALER_Y_PATH = r"C:\文件\MISO_Trading_Analysis\outputs\scaler_y.pkl"
POSITION_MW = 25    # standard virtual bid size (MW)
THRESHOLD   = 2.0   # minimum |predicted spread| to trade ($/MWh)

## 2. Load Data

In [2]:
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM hourly_features", conn)
conn.close()

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)
df = df[df['timestamp'] < '2026-01-01'].copy()

print(f"Loaded {len(df):,} rows")
print(f"Date range: {df['timestamp'].min().date()} to {df['timestamp'].max().date()}")

Loaded 26,303 rows
Date range: 2023-01-01 to 2025-12-31


## 3. Feature Selection & Train/Test Split

In [3]:
FEATURE_COLS = [
    # Temporal
    'hour', 'day_of_week', 'month', 'quarter',
    'is_peak_hour', 'is_weekend', 'is_holiday',
    # Weather & load forecasts (day-ahead available)
    'temp_forecast_f', 'forecasted_load_mw', 'wind_forecast_mw',
    # Fuel
    'gas_price',
    # Outages
    'forced_outages_mw', 'planned_outages_mw',
    'unplanned_outages_mw', 'total_outages_mw',
    # Transmission congestion
    'binding_constraints_count', 'max_shadow_price',
    # Lagged spread (autoregressive)
    'spread_lag_1h', 'spread_lag_24h', 'spread_lag_168h',
    # Rolling averages
    'spread_7day_rolling_avg', 'spread_30day_rolling_avg',
    'temp_7day_rolling_avg',
]
TARGET_COL = 'spread'

df = df.dropna(subset=FEATURE_COLS + [TARGET_COL])
print(f"Rows after dropping nulls: {len(df):,}")

train = df[df['timestamp'] < '2025-01-01'].copy()
test  = df[df['timestamp'] >= '2025-01-01'].copy()

print(f"Train: {len(train):,} ({train['timestamp'].min().date()} to {train['timestamp'].max().date()})")
print(f"Test:  {len(test):,}  ({test['timestamp'].min().date()} to {test['timestamp'].max().date()})")

Rows after dropping nulls: 24,872
Train: 16,487 (2023-01-08 to 2024-12-31)
Test:  8,385  (2025-01-01 to 2025-12-30)


## 4. Normalize & Build Sequences

In [4]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train = scaler_X.fit_transform(train[FEATURE_COLS].values)
X_test  = scaler_X.transform(test[FEATURE_COLS].values)
y_train = scaler_y.fit_transform(train[[TARGET_COL]].values)
y_test  = scaler_y.transform(test[[TARGET_COL]].values)

def build_sequences(X, y, seq_len):
    """Slide a window of seq_len over X/y to produce (samples, timesteps, features)."""
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i - seq_len:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = build_sequences(X_train, y_train, SEQUENCE_LENGTH)
X_test_seq,  y_test_seq  = build_sequences(X_test,  y_test,  SEQUENCE_LENGTH)

print(f"X_train: {X_train_seq.shape}  y_train: {y_train_seq.shape}")
print(f"X_test:  {X_test_seq.shape}   y_test:  {y_test_seq.shape}")

X_train: (16319, 168, 23)  y_train: (16319, 1)
X_test:  (8217, 168, 23)   y_test:  (8217, 1)


## 5. PyTorch Dataset & Model Definition

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Tensors
X_train_t = torch.FloatTensor(X_train_seq)
y_train_t = torch.FloatTensor(y_train_seq)
X_test_t  = torch.FloatTensor(X_test_seq).to(device)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class SpreadLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True,
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model     = SpreadLSTM(input_size=len(FEATURE_COLS), hidden_size=HIDDEN_SIZE,
                       num_layers=NUM_LAYERS, dropout=DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()

Device: cpu


## 6. Train

In [ ]:
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{EPOCHS}  Loss: {avg_loss:.4f}")

print("Training complete.")

# Save model weights and scalers
torch.save(model.state_dict(), MODEL_PATH)
joblib.dump(scaler_X, SCALER_X_PATH)
joblib.dump(scaler_y, SCALER_Y_PATH)
print(f"Saved model:    {MODEL_PATH}")
print(f"Saved scaler_X: {SCALER_X_PATH}")
print(f"Saved scaler_y: {SCALER_Y_PATH}")

Epoch  5/30  Loss: 0.8588


## 7. Evaluate on 2025 Test Set

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_t).cpu().numpy()

y_pred   = scaler_y.inverse_transform(y_pred_scaled)
y_actual = scaler_y.inverse_transform(y_test_seq)

mae     = mean_absolute_error(y_actual, y_pred)
rmse    = np.sqrt(mean_squared_error(y_actual, y_pred))
dir_acc = ((y_actual > 0) == (y_pred > 0)).mean() * 100
baseline_mae = np.abs(y_actual - y_actual.mean()).mean()

print(f"MAE:                  ${mae:.2f}  (baseline: ${baseline_mae:.2f})")
print(f"RMSE:                 ${rmse:.2f}")
print(f"Directional Accuracy: {dir_acc:.1f}%")

# Excluding extreme events (>3 std)
mask = (np.abs(y_actual - y_actual.mean()) < 3 * y_actual.std()).flatten()
print(f"\nExcluding {(~mask).sum()} extreme hours (>3σ):")
print(f"  MAE:                  ${mean_absolute_error(y_actual[mask], y_pred[mask]):.2f}")
print(f"  RMSE:                 ${np.sqrt(mean_squared_error(y_actual[mask], y_pred[mask])):.2f}")
print(f"  Directional Accuracy: {((y_actual[mask] > 0) == (y_pred[mask] > 0)).mean()*100:.1f}%")

# Quarterly breakdown
test_eval = test.iloc[SEQUENCE_LENGTH:].copy().reset_index(drop=True)
test_eval['y_pred']    = y_pred.flatten()
test_eval['y_actual']  = y_actual.flatten()
test_eval['correct_dir'] = (test_eval['y_actual'] > 0) == (test_eval['y_pred'] > 0)

quarterly = test_eval.groupby('quarter').apply(lambda g: pd.Series({
    'hours':    len(g),
    'mae':      mean_absolute_error(g['y_actual'], g['y_pred']),
    'dir_acc':  g['correct_dir'].mean() * 100,
})).round(2)

print("\nQuarterly Performance (2025):")
print(quarterly)

## 8. Signal Generation & Backtest

In [ ]:
bt = test_eval.copy()
bt['signal'] = np.where(bt['y_pred'] >  THRESHOLD,  1,
               np.where(bt['y_pred'] < -THRESHOLD, -1, 0))

# DEC profit = spread × MW;  INC profit = -spread × MW
bt['pnl'] = bt['signal'] * bt['y_actual'] * POSITION_MW

trades = bt[bt['signal'] != 0]
print(f"Trade hours:    {len(trades):,} / {len(bt):,} ({len(trades)/len(bt)*100:.1f}%)")
print(f"Win rate:       {(trades['pnl'] > 0).mean()*100:.1f}%")
print(f"Total P&L:      ${trades['pnl'].sum():,.0f}")
print(f"Avg P&L/hour:   ${trades['pnl'].mean():.2f}")

print("\nP&L by Quarter:")
print(trades.groupby('quarter')['pnl'].agg(
    total='sum', avg='mean', count='count'
).round(2))

# Threshold sensitivity
print(f"\n{'Threshold':>10} {'Trades':>8} {'Win%':>7} {'Total P&L':>12} {'Avg $/hr':>10}")
print("-" * 55)
for thr in [0.5, 1.0, 2.0, 3.0, 5.0]:
    sig = np.where(bt['y_pred'] > thr, 1, np.where(bt['y_pred'] < -thr, -1, 0))
    pnl = (sig * bt['y_actual'].values * POSITION_MW)[sig != 0]
    if len(pnl):
        print(f"{thr:>10.1f} {len(pnl):>8,} {(pnl>0).mean()*100:>6.1f}%"
              f" ${pnl.sum():>11,.0f} ${pnl.mean():>9.2f}")

## 9. Visualize Model Performance

In [ ]:
bt['cumulative_pnl'] = (bt['pnl'] * (bt['signal'] != 0)).cumsum()

sample = bt.iloc[1000:1336].copy()  # 2-week window for clarity

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Panel 1: Predicted vs actual spread
axes[0].plot(sample['timestamp'], sample['y_actual'], label='Actual',    alpha=0.7)
axes[0].plot(sample['timestamp'], sample['y_pred'],   label='Predicted', alpha=0.7, linestyle='--')
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_title('Actual vs Predicted DA-RT Spread — 2-Week Sample (2025)')
axes[0].set_ylabel('Spread ($/MWh)')
axes[0].legend()

# Panel 2: Trade signals
colors = ['green' if s > 0 else 'red' if s < 0 else 'lightgray'
          for s in sample['signal']]
axes[1].bar(sample['timestamp'], sample['signal'], color=colors)
axes[1].set_title(f'Trade Signals — Green=DEC, Red=INC, Gray=No Trade (Threshold ${THRESHOLD})')
axes[1].set_ylabel('Signal')

# Panel 3: Cumulative P&L
axes[2].plot(bt['timestamp'], bt['cumulative_pnl'], color='steelblue')
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_title(f'Cumulative P&L — 2025 Test Set (Threshold ${THRESHOLD}, {POSITION_MW} MW)')
axes[2].set_ylabel('Cumulative P&L ($)')

plt.tight_layout()
plt.savefig(rf"{OUT_PATH}\model_performance.png", dpi=150)
plt.show()
print(f"Saved: {OUT_PATH}\\model_performance.png")

## 10. SHAP Feature Importance

In [ ]:
def model_predict(X):
    """Wrapper for SHAP: accepts 2D array (samples, features), adds sequence dim."""
    X_3d = X.reshape(X.shape[0], 1, X.shape[1])
    X_t  = torch.FloatTensor(X_3d).to(device)
    with torch.no_grad():
        return model(X_t).cpu().numpy().flatten()

# Use last-timestep features from test sequences
X_test_last = X_test_seq[:, -1, :]

background   = shap.sample(X_test_last, 100)
explain_data = shap.sample(X_test_last, 500)

explainer   = shap.KernelExplainer(model_predict, background)
shap_values = explainer.shap_values(explain_data, nsamples=100)

shap.summary_plot(shap_values, explain_data, feature_names=FEATURE_COLS, show=False)
plt.savefig(rf"{OUT_PATH}\shap_summary.png", dpi=150)
plt.show()
print(f"Saved: {OUT_PATH}\\shap_summary.png")